In [0]:

# Cria um DataFrame de streaming que monitora arquivos novos na pasta de origem
df = (spark.readStream                                              # Inicia leitura em modo streaming
      .format("cloudFiles")                                          # Usa o Auto Loader: detecta novos arquivos automaticamente
      .option("cloudFiles.format", "csv")                            # Define o formato dos arquivos de origem como CSV
      .option("cloudFiles.schemaLocation", "/Volumes/retail_q/volumes/blob_source/_schema")  # Onde o Auto Loader guarda o esquema (colunas/tipos) inferido para reuso entre execuções
      .option("header", "true")                                      # A primeira linha do CSV contém os nomes das colunas
      .option("inferSchema", "true")                                 # Infere automaticamente os tipos de dados (int, string, double, etc.)
      .load("/Volumes/retail_q/volumes/blob_source/transactions_source/")  # Caminho da pasta onde os arquivos CSV chegam
)


# Escreve o streaming na tabela Delta destino, processando apenas novos arquivos a cada execução
(df.writeStream
 .option("checkpointLocation", "/Volumes/retail_q/volumes/blob_source/_checkpoint")  # Guarda progresso: evita reprocessar arquivos já lidos
 .trigger(availableNow=True)       #processa tudo que está pendente e para. Processa todos os arquivos pendentes e para (modo batch incremental)
 .toTable("retail_q.blob_bronze.transactions")                        # Tabela Delta destino (camada bronze) onde os dados são gravados
 )

In [0]:
%sql 

select count(*) from retail_q.blob_bronze.transactions